Convert data to nnUNet_raw folder format


## Dataset folder structure
Datasets must be located in the `nnUNet_raw` folder (which you either define when installing nnU-Net or export/set every
time you intend to run nnU-Net commands!).
Each segmentation dataset is stored as a separate 'Dataset'. Datasets are associated with a dataset ID, a three digit
integer, and a dataset name (which you can freely choose): For example, Dataset005_Prostate has 'Prostate' as dataset name and
the dataset id is 5. Datasets are stored in the `nnUNet_raw` folder like this:

    nnUNet_raw/
    ├── Dataset001_BrainTumour
    ├── Dataset002_Heart
    ├── Dataset003_Liver
    ├── Dataset004_Hippocampus
    ├── Dataset005_Prostate
    ├── ...

Within each dataset folder, the following structure is expected:

    Dataset001_BrainTumour/
    ├── dataset.json
    ├── imagesTr
    ├── imagesTs  # optional
    └── labelsTr


When adding your custom dataset, take a look at the [dataset_conversion](../nnunetv2/dataset_conversion) folder and
pick an id that is not already taken. IDs 001-010 are for the Medical Segmentation Decathlon.

- **imagesTr** contains the images belonging to the training cases. nnU-Net will perform pipeline configuration, training with
cross-validation, as well as finding postprocessing and the best ensemble using this data.
- **imagesTs** (optional) contains the images that belong to the test cases. nnU-Net does not use them! This could just
be a convenient location for you to store these images. Remnant of the Medical Segmentation Decathlon folder structure.
- **labelsTr** contains the images with the ground truth segmentation maps for the training cases.
- **dataset.json** contains metadata of the dataset.

The scheme introduced [above](#what-do-training-cases-look-like) results in the following folder structure. Given
is an example for the first Dataset of the MSD: BrainTumour. This dataset hat four input channels: FLAIR (0000),
T1w (0001), T1gd (0002) and T2w (0003). Note that the imagesTs folder is optional and does not have to be present.

    nnUNet_raw/Dataset001_BrainTumour/
    ├── dataset.json
    ├── imagesTr
    │   ├── BRATS_001_0000.nii.gz
    │   ├── BRATS_001_0001.nii.gz
    │   ├── BRATS_001_0002.nii.gz
    │   ├── BRATS_001_0003.nii.gz
    │   ├── BRATS_002_0000.nii.gz
    │   ├── BRATS_002_0001.nii.gz
    │   ├── BRATS_002_0002.nii.gz
    │   ├── BRATS_002_0003.nii.gz
    │   ├── ...
    ├── imagesTs
    │   ├── BRATS_485_0000.nii.gz
    │   ├── BRATS_485_0001.nii.gz
    │   ├── BRATS_485_0002.nii.gz
    │   ├── BRATS_485_0003.nii.gz
    │   ├── BRATS_486_0000.nii.gz
    │   ├── BRATS_486_0001.nii.gz
    │   ├── BRATS_486_0002.nii.gz
    │   ├── BRATS_486_0003.nii.gz
    │   ├── ...
    └── labelsTr
        ├── BRATS_001.nii.gz
        ├── BRATS_002.nii.gz
        ├── ...

Here is another example of the second dataset of the MSD, which has only one input channel:

    nnUNet_raw/Dataset002_Heart/
    ├── dataset.json
    ├── imagesTr
    │   ├── la_003_0000.nii.gz
    │   ├── la_004_0000.nii.gz
    │   ├── ...
    ├── imagesTs
    │   ├── la_001_0000.nii.gz
    │   ├── la_002_0000.nii.gz
    │   ├── ...
    └── labelsTr
        ├── la_003.nii.gz
        ├── la_004.nii.gz
        ├── ...

Remember: For each training case, all images must have the same geometry to ensure that their pixel arrays are aligned. Also
make sure that all your data is co-registered!


Aboved document was extracted from https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/dataset_format.md

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/NCKH/nnUnet")
RAW_EXTERNAL = PROJECT_ROOT / "data/raw_external/brats20"
NNUNET_RAW = PROJECT_ROOT / "data/nnUNet_raw"

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(PROJECT_ROOT))

print("Added to PYTHONPATH:", PROJECT_ROOT)

Added to PYTHONPATH: /content/drive/MyDrive/NCKH/nnUnet


In [ ]:
from scripts.convert_brats2020_to_nnunet import convert_brats2020

In [ ]:
convert_brats2020(
    raw_external_dir=RAW_EXTERNAL,
    nnunet_raw_dir=NNUNET_RAW,
    dataset_id=101
)

Found 369 training cases


100%|██████████| 369/369 [34:06<00:00,  5.55s/it]

Done. Converted 369 cases.


## 2. Save zip file

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pathlib import Path
import shutil

DATASET_NAME = "Dataset101_BraTS2020"

SRC = Path("/content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_raw") / DATASET_NAME
TMP_ROOT = Path("/content/nnUNet_raw")
DST = TMP_ROOT / DATASET_NAME

assert SRC.exists(), f"Source not found: {SRC}"

print("Copying raw dataset from Drive to Colab disk...")
TMP_ROOT.mkdir(parents=True, exist_ok=True)

if DST.exists():
    shutil.rmtree(DST)

shutil.copytree(SRC, DST)
print("Copy completed.")

Copying raw dataset from Drive to Colab disk...
Copy completed.


In [4]:
from pathlib import Path
import zipfile

DATASET_NAME = "Dataset101_BraTS2020"

SRC_DATASET = Path("/content/nnUNet_raw") / DATASET_NAME
ZIP_TMP = Path("/content") / f"{DATASET_NAME}.zip"

print("Zipping raw dataset on local Colab disk...")
print("Source:", SRC_DATASET)
print("Temp zip:", ZIP_TMP)

with zipfile.ZipFile(ZIP_TMP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in SRC_DATASET.rglob("*"):
        arcname = Path("nnUNet_raw") / p.relative_to(SRC_DATASET.parent)
        zf.write(p, arcname)

print("Local zip created successfully.")

Zipping raw dataset on local Colab disk...
Source: /content/nnUNet_raw/Dataset101_BraTS2020
Temp zip: /content/Dataset101_BraTS2020.zip
Local zip created successfully.


In [5]:
from pathlib import Path
import shutil

ZIP_TMP = Path("/content/Dataset101_BraTS2020.zip")
DST_DIR = Path("/content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_raw")
DST_DIR.mkdir(parents=True, exist_ok=True)

DST_ZIP = DST_DIR / ZIP_TMP.name

print("Copying zip to Google Drive...")
shutil.copy2(ZIP_TMP, DST_ZIP)

print("Zip saved at:", DST_ZIP)


Copying zip to Google Drive...
Zip saved at: /content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_raw/Dataset101_BraTS2020.zip


```
Drive/
└── nnUNet_raw/
    ├── Dataset101_BraTS2020/       
    └── Dataset101_BraTS2020.zip    
```